# 🧠 GIADA Task 9 — Ca_HVA su percorsi fisiologici registrati
Legge soltanto lo split `train` del dataset targeted v1.1. Le microtracce future sono un oracle diagnostico teacher-forced, non un input causale per un rollout autonomo. I candidati Task 5 restano congelati.

In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, subprocess, sys, zipfile
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_task_9');GIADA_REPO=WORK/'giada';TEACHER_REPO=WORK/'neuron_as_deep_net'
assert not GIADA_REPO.exists() and not TEACHER_REPO.exists(),'Sessione già inizializzata: usa una sessione Kaggle nuova.'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip();print({'revision':REVISION})


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]:del sys.modules[name]
import torch
assert torch.cuda.is_available(),'Task 9 richiede GPU CUDA Kaggle per i candidati congelati.'
from src.giada_teacher import ExtractedGateFormula,PhysiologicalPathConfig,run_physiological_path_diagnostic
from src.giada_teacher.physiological_voltage_paths import EXPECTED_TRANSITION_SHA256,EXPECTED_TRANSITIONS
from src.giada_teacher.voltage_path_stress import EXPECTED_TASK5_ARCHIVE_SHA256,EXPECTED_TASK5_REPORT_SHA256
prereg=json.loads((GIADA_REPO/'experiments/teacher_physiological_voltage_paths_preregistration_v1.json').read_text())
display({'gpu':torch.cuda.get_device_name(0),'sites':prereg['design']['sites'],'regimes':prereg['design']['regimes'],'source_split':'train'})


## 🔎 Individua gli input
Servono il dataset targeted v1.1 base e l'artefatto Task 5 `giada_primitive_scaling_laws`. Se Kaggle monta il dataset base come `archive.zip`, la cella estrae soltanto manifest, schema e HDF5 in `/kaggle/working` (richiede circa 6 GiB liberi).

In [ ]:
def file_sha(path):
 d=hashlib.sha256()
 with Path(path).open('rb') as h:
  for block in iter(lambda:h.read(16*1024*1024),b''):d.update(block)
 return d.hexdigest()
INPUT_ROOT=Path('/kaggle/input');override=os.environ.get('GIADA_TASK5_ARTIFACT')
candidates=[Path(override).expanduser()] if override else []
if INPUT_ROOT.is_dir():
 candidates+=list(INPUT_ROOT.rglob('giada_primitive_scaling_laws.zip'))+list(INPUT_ROOT.rglob('archive.zip'))
 candidates += [p.parent for p in INPUT_ROOT.rglob('final_report.json') if (p.parent/'frozen_scaling_checkpoints.pt').is_file()]
def exact_task5(path):
 try:return file_sha(path)==EXPECTED_TASK5_ARCHIVE_SHA256 if path.is_file() else file_sha(path/'final_report.json')==EXPECTED_TASK5_REPORT_SHA256
 except Exception:return False
TASK5_SOURCE=next((p.resolve() for p in candidates if p.exists() and exact_task5(p)),None)
assert TASK5_SOURCE is not None,'Aggiungi giada_primitive_scaling_laws.zip agli input o imposta GIADA_TASK5_ARTIFACT.'
base_override=os.environ.get('GIADA_TARGETED_DATASET')
base_candidates=[Path(base_override).expanduser()] if base_override else []
if INPUT_ROOT.is_dir():
 base_candidates += [p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower()]
 base_candidates += [p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE=next((p.resolve() for p in base_candidates if p.exists()),None)
assert BASE_SOURCE is not None,'Dataset targeted v1.1 base non trovato. Aggiungilo agli input oppure imposta GIADA_TARGETED_DATASET.'
print({'task5':str(TASK5_SOURCE),'base':str(BASE_SOURCE)})


In [ ]:
def materialize_base(source):
 source=Path(source)
 if source.is_dir():
  assert (source/'dataset_manifest.json').is_file() and (source/'state_schema.json').is_file(),'Dataset base incompleto: mancano manifest o schema.'
  return source
 destination=WORK/'extracted_base';assert not destination.exists(),f'Estrazione già presente: {destination}. Usa una sessione nuova.'
 destination.mkdir(parents=True)
 with zipfile.ZipFile(source) as archive:
  required=('dataset_manifest.json','state_schema.json','transition_dataset.h5')
  members={name:[m for m in archive.infolist() if m.filename.endswith('/'+name)] for name in required}
  assert all(len(items)==1 for items in members.values()),'Archivio base ambiguo o incompleto.'
  parents={str(Path(items[0].filename).parent) for items in members.values()}
  assert len(parents)==1,'Manifest, schema e HDF5 provengono da directory diverse.'
  for name in required:
   member=members[name][0];print({'extracting':name,'GiB':round(member.file_size/2**30,3)},flush=True)
   with archive.open(member) as reader,(destination/name).open('xb') as writer:
    copied=0;next_notice=.1
    while True:
     block=reader.read(16*1024*1024)
     if not block:break
     writer.write(block);copied+=len(block)
     if name.endswith('.h5') and copied/member.file_size>=next_notice:
      print(f'[GIADA Task 9] estrazione HDF5 {round(100*copied/member.file_size)}%',flush=True);next_notice+=.1
 return destination
BASE_ROOT=materialize_base(BASE_SOURCE)
manifest=json.loads((BASE_ROOT/'dataset_manifest.json').read_text())
assert manifest['transition_count']==EXPECTED_TRANSITIONS and manifest['teacher_commit']==prereg['source']['teacher_commit']
print({'base_root':str(BASE_ROOT),'transition_count':manifest['transition_count'],'hash_verification':'next cell'})


## 🧪 Esegui la diagnosi
La verifica SHA-256 del file HDF5 da circa 6 GiB è inclusa e mostra il progresso ogni 10%. Dopo il controllo, il notebook campiona fino a 4.000 transizioni `train`; non legge gli split di test per selezionare esempi. Il report segnala esplicitamente se la formula su 40 campioni non ricostruisce abbastanza bene i gate del teacher CVode.

In [ ]:
formula=ExtractedGateFormula.from_mod(TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
config=PhysiologicalPathConfig()
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_physiological_voltage_paths')
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Usa una sessione nuova.'
def progress(percent,label):print(f'[GIADA Task 9][SHA-256 {label}] {percent}%',flush=True)
report=run_physiological_path_diagnostic(formula,BASE_ROOT,TASK5_SOURCE,OUTPUT_DIR,config,code_revision=REVISION,progress=progress)
display({'valid':report['valid'],'selected_paths':report['support']['selected_path_count'],'support_shortfalls':report['support']['support_shortfalls'],'teacher_formula_floor':report['teacher_formula_floor'],'teacher_floor_calibrated':report['teacher_formula_floor_calibrated'],'source_split_train_only':report['source_split_train_only'],'models_retrained':report['models_retrained']})
assert report['valid'] and report['source_split_train_only'] and not report['models_retrained']


In [ ]:
rows=[]
for key,value in report['group_metrics'].items():
 rows.append({'site_regime':key,'n':value['count'],'formula_teacher_m':round(value['formula_vs_teacher']['m_rmse'],6),'formula_teacher_h':round(value['formula_vs_teacher']['h_rmse'],6),'start_only_m':round(value['vs_formula_fine']['formula_start_only']['m_rmse'],6),'coarse_m':round(value['vs_formula_fine']['formula_coarse_path']['m_rmse'],6),'lut_fine_m':round(value['vs_formula_fine']['lut_fine_path']['m_rmse'],6),'physical17_fine_m':round(value['vs_formula_fine']['physical_fine_path_seed17']['m_rmse'],6),'physical29_fine_m':round(value['vs_formula_fine']['physical_fine_path_seed29']['m_rmse'],6),'physical43_fine_m':round(value['vs_formula_fine']['physical_fine_path_seed43']['m_rmse'],6)})
import pandas as pd
display(pd.DataFrame(rows))
if not report['teacher_formula_floor_calibrated']:print('ATTENZIONE: reference floor CVode/microtraccia sopra soglia. Non interpretare candidato vs teacher; usa solo candidato vs formula e scarica il report.')


## 📦 Scarica lo ZIP con il metodo Blob/base64
Lo ZIP contiene soltanto report e indici selezionati, non copia il dataset HDF5 da 6 GiB.

In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_physiological_voltage_paths','zip',OUTPUT_DIR.parent,OUTPUT_DIR.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
